# Continuous Kalman Filter Tracking Algorithm
*Developed by 6532028621 Komkanin Maneesaisuwan ME21 (Email: komkanin.m@gmail.com)*

* The core objective of the pipeline is to enable reliable, long-duration traffic
video processing on cloud platforms like Google Colab without data loss.
* By modifying the underlying Ultralytics library and creating a new custom workflow pipeline, the system segments video processing into iterative intervals while saving and transferring the Kalman filter tracking states across boundaries.
* This architecture effectively overcomes structural hardware limitations and volatile memory constraints, mitigating tracking failures such as ID switches and fragmented trajectory trails during unexpected system interruptions or network disconnections.

# Setup
You must run this cell before anything else to mount Google Drive, install standard dependencies, and load model into memory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics
!pip install lap

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov9e.pt")

# Before running the pipeline, modify these parameters inside the cell to match your project

VIDEO_PATH: The path to your input video file in Google Drive.

OUTPUT_DIR: The folder where all output data (chunks, CSVs, logs, and tracking states) will be saved.

YAML_CONFIG: The path to your ByteTrack parameter file (controls tracking thresholds).

SAVE_INTERVAL: How often the system saves a checkpoint.

CONF & IOU: Model detection thresholds.

IMGSZ: YOLO Processing resolution

PRINT_CURRENT_FRAME: Set to False to hide frame-by-frame text logs

In [ ]:
VIDEO_PATH   = "/content/drive/MyDrive/ME21_Workspace/Makkasan_trim_20s.mp4"
OUTPUT_DIR   = "/content/drive/MyDrive/ME21_Workspace/SegmentVid/SegmentVid_Eval_250469"
YAML_CONFIG  = "/content/drive/MyDrive/ME21_Workspace/SegmentVid/SegmentVid_V2_020269/src/Makkasan.yaml"

SAVE_INTERVAL = 1000
MODEL_NAME    = "yolov9e.pt"
CONF          = 0.1
IOU           = 0.45
IMGSZ         = [1088,1920]
PRINT_CURRENT_FRAME = False

# Click to Run the Pipeline



* Install Pipeline from GitHub

* Swaps out the baseline library code with the modified script.

* Launches tracker.run()




In [ ]:
#@title Github Import
!rm -rf /content/SeniorProject-ContinuousTracking
!git clone https://github.com/KomkaninM/SeniorProject-ContinuousTracking.git SeniorProject_ContinuousTracking

In [ ]:
import importlib
import sys

import SeniorProject_ContinuousTracking.src.ContinuousTracking
importlib.reload(SeniorProject_ContinuousTracking.src.ContinuousTracking)

from SeniorProject_ContinuousTracking.src.ContinuousTracking import ContinuousTracking

In [ ]:
import os
import sys
import shutil

# --- 1. CONFIGURATION ---
# CHANGE THESE PATHS TO MATCH YOUR GOOGLE DRIVE
GITHUB_PROJECT_ROOT = "/content/SeniorProject-ContinuousTracking"

# --- 2. SETUP ENVIRONMENT ---
ULTRALYTICS_TRACKER_PATH = "/usr/local/lib/python3.12/dist-packages/ultralytics/trackers/byte_tracker.py"
CUSTOM_TRACKER_SOURCE    = os.path.join(GITHUB_PROJECT_ROOT, "src/modified_byte_tracker.py")

if os.path.exists(CUSTOM_TRACKER_SOURCE):
    shutil.copy(CUSTOM_TRACKER_SOURCE, ULTRALYTICS_TRACKER_PATH)
    print(" ByteTracker file swapped successfully!")
else:
    print(f" Warning: Could not find modified tracker at {CUSTOM_TRACKER_SOURCE}")

# --- 3. IMPORT YOUR PIPELINE ---
src_path = os.path.join(GITHUB_PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from SeniorProject_ContinuousTracking.src.ContinuousTracking import ContinuousTracking

# --- 4. RUN THE PIPELINE ---
print("\n Initializing Pipeline...")
tracker = ContinuousTracking(
    video_path = VIDEO_PATH,
    output_dir = OUTPUT_DIR,
    tracker_config_path = YAML_CONFIG,
    save_interval = SAVE_INTERVAL,
    model_name = MODEL_NAME,
    conf = CONF,
    iou = IOU,
    imgsz= IMGSZ,
    print_current_frame = PRINT_CURRENT_FRAME,
)


# Run this line to start everything.
tracker.run()